In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

In [144]:
train_path="https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"
#lecture du fichier avec la fonction read_csv de pandas
train_df=pd.read_csv(train_path)
train_df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000
mean,-119.562108,35.625225,28.589353,2643.664412,539.410824,1429.573941,501.221941,3.883578,207300.912353
std,2.005166,2.137340,12.586937,2179.947071,421.499452,1147.852959,384.520841,1.908157,115983.764387
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.790000,33.930000,18.000000,1462.000000,297.000000,790.000000,282.000000,2.566375,119400.000000
50%,-118.490000,34.250000,29.000000,2127.000000,434.000000,1167.000000,409.000000,3.544600,180400.000000
75%,-118.000000,37.720000,37.000000,3151.250000,648.250000,1721.000000,605.250000,4.767000,265000.000000
max,-114.310000,41.950000,52.000000,37937.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [179]:
# suppression des enregistrements qui contiennent des cellules vides
missing_values = train_df.isnull().sum()
if missing_values.sum() != 0:
    train_df.dropna(axis = 0)
    print("\n observation : valeurs manquantes supprime ")
else:
    print(missing_values)
    print("\n observation : aucunes valeurs manquantes trouvees ")


longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

 observation : aucunes valeurs manquantes trouvees 


In [180]:

print(train_df.columns)

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')


In [182]:
#selection des variables qui permettrons d'expliquer la variables cible
def features_selection (train_df):
    features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
           'total_bedrooms', 'population', 'households', 'median_income',]
    selected_features = train_df[features]
    proceced_features = selected_features.copy()
    proceced_features ["person_per_rooms"] = ( proceced_features["total_rooms"]/proceced_features["population"])
    return proceced_features




In [184]:
print(features_selection(train_df).head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -114.31     34.19                15.0       5612.0          1283.0   
1    -114.47     34.40                19.0       7650.0          1901.0   
2    -114.56     33.69                17.0        720.0           174.0   
3    -114.57     33.64                14.0       1501.0           337.0   
4    -114.57     33.57                20.0       1454.0           326.0   

   population  households  median_income  person_per_rooms  
0      1015.0       472.0         1.4936          5.529064  
1      1129.0       463.0         1.8200          6.775908  
2       333.0       117.0         1.6509          2.162162  
3       515.0       226.0         3.1917          2.914563  
4       624.0       262.0         1.9250          2.330128  


In [186]:
#selection et convertion de la variable cible en unite de 1000
def output_target_selection (train_df):
    output_target =pd.DataFrame()
    output_target["median_house_value"] = (train_df.median_house_value / 1000.0)
    return output_target

In [187]:
print(output_target_selection(train_df).head())

   median_house_value
0                66.9
1                80.1
2                85.7
3                73.4
4                65.5


In [190]:
X, y = features_selection(train_df), output_target_selection(train_df)
print(X,y)

       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0        -114.31     34.19                15.0       5612.0          1283.0   
1        -114.47     34.40                19.0       7650.0          1901.0   
2        -114.56     33.69                17.0        720.0           174.0   
3        -114.57     33.64                14.0       1501.0           337.0   
4        -114.57     33.57                20.0       1454.0           326.0   
...          ...       ...                 ...          ...             ...   
16995    -124.26     40.58                52.0       2217.0           394.0   
16996    -124.27     40.69                36.0       2349.0           528.0   
16997    -124.30     41.84                17.0       2677.0           531.0   
16998    -124.30     41.80                19.0       2672.0           552.0   
16999    -124.35     40.54                52.0       1820.0           300.0   

       population  households  median_income  perso

In [191]:
from sklearn.model_selection import train_test_split
# cette etape consiste a fractionner la data set a deux parties. une partie sera utilise comme donnee d'entrainement et l'autre pour la validation
train_X, val_X, train_y,val_y = train_test_split(X,y ,random_state = 0)

In [200]:
# defintion du model parfait dans lequel le meme jeu d'entrainement est utilise pour la validation
def PerfectModel (X , y):
    model_train = DecisionTreeRegressor(random_state=1 )

    model_train.fit(X, y )
    predict = model_train.predict (X )
    mae = mean_absolute_error(predict, y)
    return (mae)


In [194]:
#defintion d'un model qui ne definit pas la profondeur maximal des feuilles
def DecisionTreewithout_max_leaf (train_X, validation_X, validation_y , train_y):
    model_train = DecisionTreeRegressor(random_state=1 )

    model_train.fit(train_X, train_y )
    predict = model_train.predict (validation_X )
    mae = mean_absolute_error(predict, validation_y)
    return (mae)
    

In [195]:

#definition du model random_state est le nombre de tour qui sera fait
#train_model = DecisionTreeRegressor(max_leaf_nodes=50, random_state=0)
def DecisionTreeWith_max_leaf(train_X, validation_X, validation_y , train_y , max_leaf_nodes):
    model_train = DecisionTreeRegressor(random_state=0 , max_leaf_nodes = max_leaf_nodes)

    model_train.fit(train_X, train_y )
    predict = model_train.predict (validation_X )
    mae = mean_absolute_error(predict, validation_y)
    return (mae)




In [196]:
from sklearn.ensemble import RandomForestRegressor
def RandomForestMae(train_X, train_y, val_X, val_y):
    forest_model = RandomForestRegressor(  random_state=1)
    forest_model.fit(train_X, train_y)
    forest_predict= forest_model.predict(val_X)
    mae = mean_absolute_error(forest_predict , val_y)
    return mae
    

In [152]:
test_file_path = " https://download.mlcc.google.com/mledu-datasets/california_housing_test.csv "
california_test = pd.read_csv (test_file_path)
california_test.describe()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,3000.000000,3000.00000,3000.000000,3000.000000,3000.000000,3000.000000,3000.00000,3000.000000,3000.00000
mean,-119.589200,35.63539,28.845333,2599.578667,529.950667,1402.798667,489.91200,3.807272,205846.27500
std,1.994936,2.12967,12.555396,2155.593332,415.654368,1030.543012,365.42271,1.854512,113119.68747
min,-124.180000,32.56000,1.000000,6.000000,2.000000,5.000000,2.00000,0.499900,22500.00000
25%,-121.810000,33.93000,18.000000,1401.000000,291.000000,780.000000,273.00000,2.544000,121200.00000
50%,-118.485000,34.27000,29.000000,2106.000000,437.000000,1155.000000,409.50000,3.487150,177650.00000
75%,-118.020000,37.69000,37.000000,3129.000000,636.000000,1742.750000,597.25000,4.656475,263975.00000
max,-114.490000,41.92000,52.000000,30450.000000,5419.000000,11935.000000,4930.00000,15.000100,500001.00000


In [198]:
california_test.dropna(axis=0)
california_test.describe()



,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,3000.000000,3000.00000,3000.000000,3000.000000,3000.000000,3000.000000,3000.00000,3000.000000,3000.00000
mean,-119.589200,35.63539,28.845333,2599.578667,529.950667,1402.798667,489.91200,3.807272,205846.27500
std,1.994936,2.12967,12.555396,2155.593332,415.654368,1030.543012,365.42271,1.854512,113119.68747
min,-124.180000,32.56000,1.000000,6.000000,2.000000,5.000000,2.00000,0.499900,22500.00000
25%,-121.810000,33.93000,18.000000,1401.000000,291.000000,780.000000,273.00000,2.544000,121200.00000
50%,-118.485000,34.27000,29.000000,2106.000000,437.000000,1155.000000,409.50000,3.487150,177650.00000
75%,-118.020000,37.69000,37.000000,3129.000000,636.000000,1742.750000,597.25000,4.656475,263975.00000
max,-114.490000,41.92000,52.000000,30450.000000,5419.000000,11935.000000,4930.00000,15.000100,500001.00000


In [216]:
validation_X, validation_y = features_selection(california_test), output_target_selection(california_test)
validation_y.describe()

,median_house_value
count,3000.000000
mean,205.846275
std,113.119687
min,22.500000
25%,121.200000
50%,177.650000
75%,263.975000
max,500.001000


In [204]:
#cette boucle nous permet de tester differentes profondeurs de feuilles afin de  trouver la mieux adapter
for max_leaf_nodes in [10,50,100,200,250,260,270,500,550,600]:
    my_error = Mae(train_X, validation_X, validation_y,train_y ,max_leaf_nodes)
    print("max leaf :%d \t\t l'erreur est de: %d " %(max_leaf_nodes, my_error))

max leaf :10 		 l'erreur est de: 58 
max leaf :50 		 l'erreur est de: 45 
max leaf :100 		 l'erreur est de: 42 
max leaf :200 		 l'erreur est de: 40 
max leaf :250 		 l'erreur est de: 39 
max leaf :260 		 l'erreur est de: 39 
max leaf :270 		 l'erreur est de: 39 
max leaf :500 		 l'erreur est de: 38 
max leaf :550 		 l'erreur est de: 38 
max leaf :600 		 l'erreur est de: 38 


In [206]:
print("Validation du Mae avec le model parfait : {:,.0f}".format(PerfectModel(X, y)))
print("Validation du Mae avec l'abres de decision sans priciser la profondeur : {:,.0f}".format(DecisionTreewithout_max_leaf(train_X, validation_X, validation_y , train_y)))
print("Validation du Mae avec l'abres de decision avec la profondeur : {:,.0f}".format(DecisionTreeWith_max_leaf(train_X, validation_X, validation_y , train_y , 500)))
print("Validation du Mae avec la forect aleatoire : {:,.0f}".format(RandomForestMae(train_X,train_y ,validation_X, validation_y)))

Validation du Mae avec le model parfait : 0
Validation du Mae avec l'abres de decision sans priciser max_leave_node : 42
Validation du Mae avec l'abres de decision avec  max_leave_node : 39


C:\Python312\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Validation du Mae avec la forect aleatoire : 31
